# Depressive and Suicidal text classification

**Datasets used:**
1. [Classified Twitter posts](https://www.kaggle.com/datasets/munkialbright/classified-tweets)
2. [Derived dataset from Murarka et al., Nikhileswar Komati and Suchintika Sarkar](https://www.kaggle.com/datasets/priyangshumukherjee/mental-health-text-classification-dataset)

## Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

## Download the dataset

In [2]:
import kagglehub

In [3]:
path1 = kagglehub.dataset_download("munkialbright/classified-tweets")
print("Path to dataset 1:", path1)

Using Colab cache for faster access to the 'classified-tweets' dataset.
Path to dataset 1: /kaggle/input/classified-tweets


In [4]:
path2 = kagglehub.dataset_download("priyangshumukherjee/mental-health-text-classification-dataset")
print("Path to dataset 2:", path2)

Using Colab cache for faster access to the 'mental-health-text-classification-dataset' dataset.
Path to dataset 2: /kaggle/input/mental-health-text-classification-dataset


## Set up dataframes

In [5]:
# We do multi-label (binary label) classification
# with possible missing values in the labels
# 0 = not present, 1 = present, -1 = unknown/missing
# normal text have all labels = 0

df_all = pd.DataFrame(columns=["text", "isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"])

In [6]:
df_all1 = pd.read_csv(path1 + "/classified_tweets.csv")
# print columns
print("Columns in dataset 1:", df_all1.columns)

# cols = 
# 'text' -> text of the tweet 
# 'suspicious' -> 0 or 1 if the tweet is suspicious (contains harmful content)
# 'cyberbullying' -> 0 if nothing, 1 if racist, 2 if sexist
# 'hate' -> 0 or 1 if the tweet contains hate speech
# 'suicidal' -> 0 or 1 if the tweet contains suicidal content

# this dataset has no depressive / anxiety labels, so we will fill them with -1 (unknown/missing)
df_all1["isDepressive"] = -1
df_all1["isAnxiety"] = -1
# rename columns to match our format
df_all1.rename(columns={
    "suspicious": "isSuspicious", 
    "hate": "isHate", 
    "suicidal": "isSuicidal"
}, inplace=True)

# convert cyberbullying to isRacist and isSexist
df_all1["isRacist"] = df_all1["cyberbullying"].apply(lambda x: 1 if x == 1 else 0)
df_all1["isSexist"] = df_all1["cyberbullying"].apply(lambda x: 1 if x == 2 else 0)
# drop the original cyberbullying column
df_all1.drop(columns=["cyberbullying"], inplace=True)

display(df_all1.head())

df_all = pd.concat([df_all, df_all1], ignore_index=True)

# debug, print unique values of row isSexist
print("Unique values in isSexist:", df_all["isSexist"].unique())
# count how many rows have isSexist = 1
print("Number of rows with isSexist = 1:", (df_all["isSexist"] == 1).sum())

Columns in dataset 1: Index(['text', 'suspicious', 'cyberbullying', 'hate', 'suicidal'], dtype='object')


,text,isSuspicious,isHate,isSuicidal,isDepressive,isAnxiety,isRacist,isSexist
0,Uhmm like 6th grade on a corner of a street....,0,0,0,-1,-1,0,0
1,a) JTP is a douchebag b) Stewart kicks ass!,1,0,0,-1,-1,0,0
2,ditto bitch!,1,0,0,-1,-1,0,0
3,damn I have to drive my dad to the airport tha...,0,0,0,-1,-1,0,0
4,:],0,0,0,-1,-1,0,0


Unique values in isSexist: [0 1]
Number of rows with isSexist = 1: 1733


In [7]:
df_all2 = pd.read_csv(path2 + "/mental_heath_unbanlanced.csv")

# print columns
print("Columns in dataset 2:", df_all2.columns)

# cols =
# 'text' -> text of the post
# 'status' -> the text Normal, Depression, Suicidal, Anxiety

def getRow(
        default_num : int,
        change_key : str = None,
        change_value : int = None
    ):
    res = {
        "isSuspicious": default_num, 
        "isRacist": default_num, 
        "isSexist": default_num, 
        "isHate": default_num, 
        "isDepressive": default_num, 
        "isSuicidal": default_num, 
        "isAnxiety": default_num,
    }
    if change_key is not None and change_value is not None:
        res[change_key] = change_value
    return res

mapping_cols = {
    "Normal"     : getRow(0),
    "Depression" : getRow(-1, "isDepressive", 1),
    "Suicidal"   : getRow(-1, "isSuicidal", 1),
    "Anxiety"    : getRow(-1, "isAnxiety", 1),
}

# apply the mapping to the status column and create new columns
for status, mapping in mapping_cols.items():
    df_all2.loc[df_all2["status"] == status, ["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]] = mapping.values()

# drop the original status column
df_all2.drop(columns=["status", "Unique_ID"], inplace=True)

display(df_all2.head())

df_all = pd.concat([df_all, df_all2], ignore_index=True)

Columns in dataset 2: Index(['Unique_ID', 'text', 'status'], dtype='object')


,text,isSuspicious,isRacist,isSexist,isHate,isDepressive,isSuicidal,isAnxiety
0,oh my gosh,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0
1,"trouble sleeping, confused mind, restless hear...",-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0
2,"All wrong, back off dear, forward doubt. Stay ...",-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0
3,I've shifted my focus to something else but I'...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0
4,"I'm restless and restless, it's been a month n...",-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0


In [8]:
del df_all1
del df_all2

In [9]:
# turn any value > 1 in whatever col into 1
for col in ["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]:
    df_all[col] = df_all[col].apply(lambda x: 1 if x > 1 else x)

## Print stat

In [10]:
num_samples = len(df_all)
print("Total number of samples:", num_samples)

Total number of samples: 69546


In [11]:
count_df = df_all[["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]].apply(pd.Series.value_counts).T
print("Label distribution:")
print(count_df)

Label distribution:
               -1.0    0.0    1.0
isSuspicious  31221  31110   7215
isRacist      31221  37380    945
isSexist      31221  36592   1733
isHate        31221  35647   2678
isDepressive  36649  18391  14506
isSuicidal    20009  37277  12260
isAnxiety     45652  18391   5503


## Helpers

In [12]:
from sklearn.model_selection import train_test_split
# split into train and test sets
train_df, test_df = train_test_split(df_all, test_size=0.2, random_state=42)

In [13]:
score_df = pd.DataFrame(columns=["model", "precision", "recall", "f1-score"])

In [14]:
from sklearn.metrics import classification_report, multilabel_confusion_matrix, accuracy_score, f1_score, precision_score, recall_score

def evaluate(
    model_name : str,
    preditor, # function from tensor of features (X) to list of predicted labels
    encoder, # function from list of texts to tensor of features (X)
):
    X = encoder(test_df["text"].tolist())
    y_true = test_df[["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]].values
    y_pred = preditor(X)
    # For missing labels (-1), consider any prediction as correct
    missing_mask = y_true == -1
    y_true = y_true.astype(float)
    y_true[missing_mask] = y_pred[missing_mask]
    y_true = y_true.astype(int)
    # print classification report for each label
    print(classification_report(y_true, y_pred, target_names=["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]))
    # print multilabel confusion matrix for each label
    # print(multilabel_confusion_matrix(y_true, y_pred))
    # print overall accuracy, f1 score, precision, recall
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")

    print("Overall accuracy:", accuracy)
    print("Overall f1 score:", f1)
    print("Overall precision:", precision)
    print("Overall recall:", recall)

    score_df.loc[len(score_df)] = [model_name, precision, recall, f1]

In [15]:
def train_and_test(
    models, # list of (model_name, model (with predict method)),
    encoders, # list of (encoder_name, encoder models (with fit and transform methods))
):
    # fit
    encoded = {} # map from encoder_name to encoded X_train
    for encoder_name, encoder in encoders:
        print(f"Fitting encoder: {encoder_name}...")
        # y_train = train_df[["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]].values
        X_train = encoder.fit_transform(train_df["text"].tolist())
        encoded[encoder_name] = X_train

    Y_train = train_df[["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]].values
    # treat missing value as random between 0 and 1
    Y_train[Y_train == -1] = np.random.randint(0, 2, size=(Y_train == -1).sum())
    for model_name, model in models:
        print(f"Evaluating model: {model_name}...")
        for encoder_name, encoder in encoders:
            X_train = encoded[encoder_name]
            model.fit(X_train, Y_train)
            def predictor(X):
                X_encoded = encoder.transform(X) if isinstance(X, list) else X
                return model.predict(X_encoded)
            evaluate(f"{model_name} ({encoder_name})", predictor, lambda texts: texts)
        del model

## Dummy model

In [16]:
def random_predictor(X):
    if isinstance(X, torch.Tensor):
        X = X.cpu().numpy()
        num_samples = X.shape[0]
    else:
        if isinstance(X, list):
            num_samples = len(X)
    num_labels = 7
    return np.random.randint(2, size=(num_samples, num_labels))

evaluate("Random Predictor", random_predictor, lambda texts: texts)

              precision    recall  f1-score   support

isSuspicious       0.55      0.85      0.67      4586
    isRacist       0.46      0.97      0.63      3331
    isSexist       0.48      0.95      0.64      3542
      isHate       0.49      0.93      0.64      3708
isDepressive       0.74      0.78      0.76      6676
  isSuicidal       0.46      0.72      0.56      4430
   isAnxiety       0.74      0.90      0.81      5711

   micro avg       0.56      0.86      0.68     31984
   macro avg       0.56      0.87      0.67     31984
weighted avg       0.59      0.86      0.69     31984
 samples avg       0.56      0.60      0.55     31984

Overall accuracy: 0.2359453630481668
Overall f1 score: 0.6879984010831193
Overall precision: 0.5881843626833435
Overall recall: 0.858835667833917


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Classic models 

This section will uses traditional encoders (TF-IDF and BOW) with Traditional models (Logistic regression and Decision tree)

In [17]:
# import TF-IDF vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
# import Logistic Regression model
from sklearn.linear_model import LogisticRegression
# import MultiOutputClassifier for multi-label classification
from sklearn.multioutput import MultiOutputClassifier

In [18]:
vectorizer_tfidf = TfidfVectorizer(max_features=5000)
vectorizer_BOW = CountVectorizer(max_features=5000)

model_linear = MultiOutputClassifier(LogisticRegression(max_iter=1000))

train_and_test(
    models=[
        ("Logistic Regression", model_linear),
    ],
    encoders=[
        ("TF-IDF", vectorizer_tfidf),
        ("BOW", vectorizer_BOW),
    ]
)

Fitting encoder: TF-IDF...
Fitting encoder: BOW...
Evaluating model: Logistic Regression...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

isSuspicious       0.92      0.77      0.84      3583
    isRacist       0.98      0.96      0.97      2345
    isSexist       0.98      0.92      0.95      2545
      isHate       0.97      0.89      0.93      2618
isDepressive       0.97      0.93      0.95      6257
  isSuicidal       0.98      0.86      0.92      4534
   isAnxiety       0.98      0.96      0.97      4324

   micro avg       0.97      0.90      0.93     26206
   macro avg       0.97      0.90      0.93     26206
weighted avg       0.97      0.90      0.93     26206
 samples avg       0.51      0.49      0.50     26206

Overall accuracy: 0.8220704529115744
Overall f1 score: 0.9334051708074985
Overall precision: 0.9688255895117232
Overall recall: 0.901663741127986
              precision    recall  f1-score   support

isSuspicious       0.90      0.73      0.81      3450
    isRacist       0.96      0.96      0.96      2123
    isSexist       0.97      0.92      0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [19]:
display(score_df)

,model,precision,recall,f1-score
0,Random Predictor,0.588184,0.858836,0.687998
1,Logistic Regression (TF-IDF),0.968826,0.901664,0.933405
2,Logistic Regression (BOW),0.952583,0.864542,0.905065
